## Homework

> Note: sometimes your answer doesn't match one of the options exactly. 
> That's fine. 
> Select the option that's closest to your solution.


### Dataset

In this homework, we will use the lead scoring dataset Bank Marketing dataset. Download it from [here](https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv).

Or you can do it with `wget`:

```bash
wget https://raw.githubusercontent.com/alexeygrigorev/datasets/master/course_lead_scoring.csv
```

In this dataset our desired target for classification task will be `converted` variable - has the client signed up to the platform or not. 

### Data preparation

* Check if the missing values are presented in the features.
* If there are missing values:
    * For caterogiral features, replace them with 'NA'
    * For numerical features, replace with with 0.0 

### Question 1

What is the most frequent observation (mode) for the column `industry`?

- `NA`
- `technology`
- `healthcare`
- **`retail`**

In [26]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

In [27]:
df = pd.read_csv('../data/course_lead_scoring.csv')
df.head()

,lead_source,industry,number_of_courses_viewed,annual_income,employment_status,location,interaction_count,lead_score,converted
0,paid_ads,NaN,1,79450.0,unemployed,south_america,4,0.94,1
1,social_media,retail,1,46992.0,employed,south_america,1,0.80,0
2,events,healthcare,5,78796.0,unemployed,australia,3,0.69,1
3,paid_ads,retail,2,83843.0,NaN,australia,1,0.87,0
4,referral,education,3,85012.0,self_employed,europe,3,0.62,1


In [28]:
target_col = "converted"
cat_cols = ["lead_source", "industry", "employment_status", "location"]
num_cols = ["number_of_courses_viewed", "annual_income", "interaction_count", "lead_score"]

In [29]:
missing_summary = df[cat_cols + num_cols].isna().sum()
missing_summary

lead_source                 128
industry                    134
employment_status           100
location                     63
number_of_courses_viewed      0
annual_income               181
interaction_count             0
lead_score                    0
dtype: int64

In [30]:
df[cat_cols] = df[cat_cols].fillna("NA")
df[num_cols] = df[num_cols].fillna(0.0)

In [31]:
q1_mode_industry = df["industry"].mode(dropna=False)[0]
print("Q1 - Mode of 'industry':", q1_mode_industry)

Q1 - Mode of 'industry': retail


### Question 2

Create the [correlation matrix](https://www.google.com/search?q=correlation+matrix) for the numerical features of your dataset. 
In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- `interaction_count` and `lead_score`
- `number_of_courses_viewed` and `lead_score`
- `number_of_courses_viewed` and `interaction_count`
- **`annual_income` and `interaction_count`**

Only consider the pairs above when answering this question.

In [32]:
corr = df[num_cols].corr()

pairs_to_check = [
    ("interaction_count", "lead_score"),
    ("number_of_courses_viewed", "lead_score"),
    ("number_of_courses_viewed", "interaction_count"),
    ("annual_income", "interaction_count"),
]

pair_corrs = {pair: abs(corr.loc[pair[0], pair[1]]) for pair in pairs_to_check}
q2_best_pair = max(pair_corrs.items(), key=lambda kv: kv[1])[0]
print("Q2 - Pair with largest correlation (among given):", q2_best_pair, "->", pair_corrs[q2_best_pair])


Q2 - Pair with largest correlation (among given): ('annual_income', 'interaction_count') -> 0.027036472404814396


### Split the data

* Split your data in train/val/test sets with 60%/20%/20% distribution.
* Use Scikit-Learn for that (the `train_test_split` function) and set the seed to `42`.
* Make sure that the target value `y` is not in your dataframe.

In [33]:
df_temp, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_temp, test_size=0.25, random_state=42)

X_train = df_train.reset_index(drop=True)
X_val = df_val.reset_index(drop=True)
X_test = df_test.reset_index(drop=True)

In [34]:
len(X_train), len(X_val), len(X_test)

(876, 293, 293)

In [35]:
y_train = X_train.converted.values
y_val = X_val.converted.values
y_test = X_test.converted.values

In [36]:
len(y_train), len(y_val), len(y_test)

(876, 293, 293)

In [37]:
del df_train['converted']
del df_val['converted']
del df_test['converted']

### Question 3

* Calculate the mutual information score between `y` and other categorical variables in the dataset. Use the training set only.
* Round the scores to 2 decimals using `round(score, 2)`.

Which of these variables has the biggest mutual information score?
  
- `industry`
- `location`
- **`lead_source`**
- `employment_status`

In [ ]:
X_train_cat = X_train[cat_cols].copy()

# Label-encode each categorical column separately
encoders = {}
X_train_cat_le = pd.DataFrame(index=X_train_cat.index)
for c in cat_cols:
    le = LabelEncoder()
    X_train_cat_le[c] = le.fit_transform(X_train_cat[c])
    encoders[c] = le

# Compute MI
mi_scores = mutual_info_classif(X_train_cat_le.values, y_train, discrete_features=True, random_state=42)
mi_by_col = {c: round(s, 2) for c, s in zip(cat_cols, mi_scores)}
q3_best_cat = max(mi_by_col.items(), key=lambda kv: kv[1])[0]

print("Q3 - Mutual information (rounded to 2 decimals):", mi_by_col)
print("Q3 - Highest MI categorical feature:", q3_best_cat)

Q3 - Mutual information (rounded to 2 decimals): {'lead_source': np.float64(0.04), 'industry': np.float64(0.01), 'employment_status': np.float64(0.01), 'location': np.float64(0.0)}
Q3 - Highest MI categorical feature: lead_source


### Question 4

* Now let's train a logistic regression.
* Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
* Fit the model on the training dataset.
    - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
    - `model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)`
* Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.64
- 0.74
- 0.84
- 0.94

In [39]:
def make_pipeline(C=1.0, drop_features=None):
    # Decide which columns to use
    used_cat = [c for c in cat_cols if (drop_features is None or c not in drop_features)]
    used_num = [c for c in num_cols if (drop_features is None or c not in drop_features)]

    preproc = ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore"), used_cat),
            ("num", "passthrough", used_num),
        ]
    )

    model = LogisticRegression(solver="liblinear", C=C, max_iter=1000, random_state=42)
    pipe = Pipeline(steps=[("preproc", preproc), ("logreg", model)])
    return pipe, used_cat, used_num

# --------------------------
# Q4: Train LR (C=1.0) with OHE, report val accuracy (round to 2 decimals)
# --------------------------
pipe_full, used_cat_full, used_num_full = make_pipeline(C=1.0)
pipe_full.fit(X_train, y_train)
y_val_pred = pipe_full.predict(X_val)
val_acc_full = accuracy_score(y_val, y_val_pred)
print("Q4 - Validation accuracy (C=1.0):", round(val_acc_full, 2))

Q4 - Validation accuracy (C=1.0): 0.7


### Question 5 

* Let's find the least useful feature using the *feature elimination* technique.
* Train a model using the same features and parameters as in Q4 (without rounding).
* Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
* For each feature, calculate the difference between the original accuracy and the accuracy without the feature. 

Which of following feature has the smallest difference?

- `'industry'`
- `'employment_status'`
- **`'lead_score'`**

> **Note**: The difference doesn't have to be positive.

In [40]:
features_to_test = ["industry", "employment_status", "lead_score"]
diffs = {}

for f in features_to_test:
    pipe_drop, _, _ = make_pipeline(C=1.0, drop_features=[f])
    pipe_drop.fit(X_train, y_train)
    y_val_pred_drop = pipe_drop.predict(X_val)
    acc_drop = accuracy_score(y_val, y_val_pred_drop)
    diffs[f] = val_acc_full - acc_drop  # could be negative

print("Q5 - Accuracy differences (full - without feature):", {k: round(v, 4) for k, v in diffs.items()})
q5_smallest_diff = min(diffs.items(), key=lambda kv: kv[1])[0]
print("Q5 - Feature with smallest difference:", q5_smallest_diff)

Q5 - Accuracy differences (full - without feature): {'industry': 0.0, 'employment_status': 0.0034, 'lead_score': -0.0068}
Q5 - Feature with smallest difference: lead_score


### Question 6

* Now let's train a regularized logistic regression.
* Let's try the following values of the parameter `C`: `[0.01, 0.1, 1, 10, 100]`.
* Train models using all the features as in Q4.
* Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these `C` leads to the best accuracy on the validation set?

- 0.01
- 0.1
- 1
- 10
- 100

> **Note**: If there are multiple options, select the smallest `C`.

In [41]:
Cs = [0.01, 0.1, 1, 10, 100]
c_to_acc = {}

for C in Cs:
    pipe_c, _, _ = make_pipeline(C=C)
    pipe_c.fit(X_train, y_train)
    pred = pipe_c.predict(X_val)
    acc = accuracy_score(y_val, pred)
    c_to_acc[C] = acc

# Pick best accuracy; tie broken by smaller C
best_acc = max(c_to_acc.values())
best_C = min([C for C, acc in c_to_acc.items() if acc == best_acc])

print("Q6 - Validation accuracies by C (rounded to 3 decimals):", {C: round(a, 3) for C, a in c_to_acc.items()})
print("Q6 - Best C:", best_C, "with val acc:", round(c_to_acc[best_C], 3))

# --------------------------
# Final answers recap (rounded per instructions)
# --------------------------
print("\n=== Answers to Submit ===")
print("Q1:", q1_mode_industry)
print("Q2:", q2_best_pair)
print("Q3:", q3_best_cat)
print("Q4:", round(val_acc_full, 2))
print("Q5:", q5_smallest_diff)
print("Q6:", best_C)

Q6 - Validation accuracies by C (rounded to 3 decimals): {0.01: 0.7, 0.1: 0.7, 1: 0.7, 10: 0.7, 100: 0.7}
Q6 - Best C: 0.01 with val acc: 0.7

=== Answers to Submit ===
Q1: retail
Q2: ('annual_income', 'interaction_count')
Q3: lead_source
Q4: 0.7
Q5: lead_score
Q6: 0.01
